# Exercise 1. Prompting!
LLMs generate words (tokens) probabilistically in various ways, depending on the architecture and sampling method. As a consequence, the we words choose to provide the LLM impact what it generates. 

The process of "choosing the right words"  is called `prompt engineering`

:::{admonition} PAPER SPOTLIGHT: {cite:t}`hedderich-etal-2025-whats`
:class: fuchsia, dropdown
If you're interested in learning more about prompting, I suggest reading:   
[What’s the Difference? Supporting Users in Identifying the Effects of
Prompt and Model Changes Through Token Patterns](https://aclanthology.org/2025.acl-long.985.pdf)" by {cite:t}`hedderich-etal-2025-whats`

This paper presents "Spotlight", an approach to identify token patterns in prompts to make `prompt-engineering` more transparent for users. Paper was presented in at one of biggest NLP conferences `ACL2025`!
:::

## 1.1 Setup: Import Packages
If you have not already, please download the packages below (in venv or in UCloud) in your terminal:

```bash
pip install transformers torch
```

:::{admonition} Or download in notebook ... 
:class: tip, dropddown Remember, you can also download the packages in Jupyter notebooks with the %pip magic command as we have done in previous classes. 
:::

Import the packages

In [2]:
from transformers import AutoTokenizer, pipeline
import transformers 
import torch 

## 1.2 Model Introduction
Let’s load Google’s `Flan-T5-base` and OpenAI’s `GPT-2`.

`Flan-T5-base` is an instruction-tuned version of Google’s influential [T5](https://huggingface.co/docs/transformers/en/model_doc/t5). It can follow prompts fairly well. However, as you will see, it does not behave quite like modern chatbots.


```{figure} ../figures/class6/flan-t5.png
---
name: flan-t5
width: 90%
---
Figure by {cite:t}`chung_scaling_2024`
```


`GPT-2`, in contrast, is **not** instruction-tuned, but is a highly influential generative model which laid the foundation for ChatGPT!

Before continuing, take a moment to think about their underlying architectures, perhaps this will make it clear why `Flan-T5` is a bit special!

:::{admonition} QUESTION
:class: red
Which component(s) make up the architecture of these models? Decoder-only? Encoder-decoder? Encoder-only? 

Discuss with a friend and google it if you don't know, then check the answers below.

<details>
<summary>ANSWER</summary>
GPT2 is decoder-only! (<a href="https://jalammar.github.io/illustrated-gpt2/">See here</a>)     

Flan-T5 is encoder-decoder (<a href = "https://huggingface.co/docs/transformers/en/model_doc/t5">See here</a>)
</details>
:::

### Load Models
We'll use `transformers.pipeline` to load the models:

In [3]:
max_new_tokens = 200 # max tokens to generate, feel free to change

When loading `FLAN-T5`, we specify the task `text2text-generation`:

In [4]:
model_id = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
pipeline_t5 = pipeline(
    task = "text2text-generation",
    model=model_id,
    dtype=torch.float16, # a way to reduce memory
    max_new_tokens=max_new_tokens,
    device_map="auto",
)

Device set to use mps


We did not set any custom hyperparameters, but let's check the standard ones. By default, `num_beams` is 4, telling us that the [beam search](https://huggingface.co/docs/transformers/en/generation_strategies#beam-search) is generating 4 options per step:

In [5]:
pipeline_t5.generation_config

GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "max_new_tokens": 200,
  "num_beams": 4,
  "pad_token_id": 0
}

For `GPT-2`, the task is `text-generation`. We set a few hyper-parameters:

In [6]:
model_id = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
pipeline_gpt = pipeline(
    task = "text-generation",
    model=model_id,
    dtype=torch.float16,
    max_new_tokens=max_new_tokens,
    return_full_text=False, 
    device_map="auto",
    temperature=0.8, 
    top_k=40,
    top_p=0.9,
)

Device set to use mps


We have set our `temperature` to `0.8` and also the `top_k` and `top_p`. You'll get to play with these in a second to get a understanding of what they do!

:::{admonition} QUESTION
:class: red
Why does `Flan-T5` use `text2text-generation` while `GPT-2` uses `text-generation`? Consider their model architectures!

<details>
<summary>ANSWER</summary>
Text-to-text generation, also called <code>sequence-to-sequence modeling</code>, maps an input sequence to an output sequence. It uses an <code>encoder–decoder</code> architecture, which processes information in both directions.
<br><br>
Text generation, on the other hand, generates text from left to right using a <code>decoder-only</code> architecture, as in GPT-2.
</details>
:::

### Your Turn: Play with Decoding Strategies!

:::{admonition} HANDS ON
:class: red
This exercise leads you to two different websites, so we'll have to briefly leave the code: 
1. To get a better understanding of `beam search`, try this [vizualizer](https://huggingface.co/spaces/m-ric/beam_search_visualizer) 
2. For the `temperature`/`top-k` sampling, go to this [demo](https://artefact2.github.io/llm-sampling/index.xhtml). Focus on the parameters we set above. 

*Note: you need to first "activate" the parameter by clicking the checkmark for the sampling demo!*
:::

## 1.3 Text Completion
Let's try to ask Flan-T5 a simple question:

In [7]:
pipeline_t5("What is the capital of Denmark?")

[{'generated_text': 'djurgrden'}]

The generated text above was obviously not what we wanted, let's phrase it in another way:

In [8]:
pipeline_t5("The capital of Denmark is")

[{'generated_text': 'Copenhagen'}]

:::{admonition} QUESTION
:class: red
Do you have any idea why the first phrasing, but the second one worked?

<details>
<summary>ANSWER</summary>
A guess would be that this has to do with the model’s default mode being <em>text completion</em> (before further fine-tuning). In text completion mode, <code>Copenhagen</code> might be a high-likelihood continuation. 
<br><br>
Fine-tuning for question answering introduces a wider range of question types and topics, which may not always include knowledge specific to Denmark, so the model may not assign as high a probability to <code>Copenhagen</code> in that context and phrasing.

</details>
:::

### Your Turn: Task Prefix & Suffix
What if we *really* want the question format to work in this context? Try it yourself:
:::{admonition} HANDS-ON
:class: red
Create a better prompt!
* Write an instruction in the `task_prefix` below to make the prompt more complete to hopefully get a better answer. You could also consider a `task_suffix`
* You can also experiment with the phrasing of the question, but keep it as a question!
* You may not be able to hit `Copenhagen`, but just focus on making it more meaningful than `djurgden`. Can we get closer to Denmark?
:::

Only task prefix:

In [9]:
task_prefix = ""
prompt = f"{task_prefix}: What is the capital of Denmark?" # pass the prompt to pipeline_t5()

Both a task prefix and suffix:

In [10]:
task_prefix = ""
task_suffix = ""
prompt = f"{task_prefix}: What is the capital of Denmark? {task_suffix}" # pass the prompt to pipeline_t5()

#### My Solution

In [11]:
task_prefix = "Answer the question appropriately"
task_suffix = "Your answer"
prompt = f"{task_prefix}: What is the capital of Denmark? {task_suffix}:" # pass the prompt to pipeline_t5()
print(prompt)
print(pipeline_t5(prompt))
# when I remove "appropriately", the model outputs "djurgden" again ...

prompt = "I was wondering what the capital of Denmark is? Can you help me answer?"
print("\n")
print(prompt)
print(pipeline_t5(prompt))

Answer the question appropriately: What is the capital of Denmark? Your answer:
[{'generated_text': 'Copenhagen'}]


I was wondering what the capital of Denmark is? Can you help me answer?
[{'generated_text': 'Copenhagen'}]


## 1.4 Summarization
A very useful application of LLMs is summarization! Let's say consider this student essay from [Class 1](../book/class1/001_simple_tokenization.ipynb):

In [12]:
student_text = """A matter of considerable controversy at present is the issue of whether distance-learning should be promoted as much as possible, or rather attending lectures in person should be allowed to take a predominant place in universities because this way of learning is superior than online degrees. From my perspective, online-teaching should be widely used among colleges and universities in terms of convenience and optimizing cost for educational institutions.
To begin with, distance-learning brings significant convenience for students in every corner of the world. In earlier times, students who were tired of commuting had to attend schools nearby, regardless of any differences in teaching facilities, teacher's qualifications or the school reputation. Now, however, students are able to apply for online-courses provided by top-of-the-range universities and colleges worldwide. 
Furthermore, the presence of video conferencing allows a teacher to teach a greater number of students. Consequently, educational institutions are able to optimize costs by increasing teacher-student ratios. Thanks to the economical online-teaching, universities and colleges are able to offer grants for students who have outstanding academic achievements but are unable to attend schools because of financial constrains.
Nevertheless, opponents of online-degrees would argue that attending lectures in person provides students an opportunity to communicate with teachers and other classmates. They further point out that traditional teaching approaches involving discussion and cooperation among students play a significant role in campus life. It is the real interactions and communications in class make education much more attractive.
By way of conclusion, it is my belief that distance-learning will become increasingly important in the future as the pace of life increases. However, discussions and interactions should be held via video conferencing frequently so that the joy of learning would not be diminished.
"""

### Your Turn: Prompt-Engineering II
:::{admonition} HANDS-ON
:class: red
- Create at least two different prompts called `prompt` (or `prompt1` / `prompt2`) for summarization by adding different `task_prefix` to `student_text` as done above.
- You can use the f-string formatting that we did previously (see explanation below). 

Try it with both `GPT-2` and `Flan-T5`!

Note: You are also allowed to find a different text than `student_text` if you prefer that!
:::

:::{admonition} What is an f-string?
:class: tip, dropdown
You have probably come across f-strings before this class. F-strings, or *formatted string literals*, let you insert variables directly into strings. To use them, place an `f` before the opening quotation mark and include the variable inside curly brackets `{}`:
```python
my_name = "Mina"
introduction = f"My name is {my_name}"

print(introduction) 
# outputs "My name is Mina"
```

Ypu can do this with as many variables you would like:
```python
my_name = "Mina"
my_city = "Aarhus" 
my_color = "green"
introduction = f"My name is {my_name} and I am from {my_city}. My favorite color is {my_color}."

print(introduction) 
# outputs "My name is Mina and I am from Aarhus. My favorite color is green."
```

As an alternative, you can also use the `format` method on a string:
```python
my_name = "Mina"
my_city = "Aarhus" 
my_color = "green"
introduction = "My name is {} and I am from {}. My favorite color is {}.".format(my_name, my_city, my_color)
# outputs "My name is Mina and I am from Aarhus. My favorite color is green."
```

You can also add two strings together:
```python
title = "The story of my life"
content = "I live in Aarhus, Denmark and have done so for many many years. I quite like it here. I also teach NLP which is SO meaningful to me!"

full_story = f"{title}: {content}"
print(full_story)
```
:::


:::{admonition} QUESTION
:class: red
How did your prompts work? Was one model or prompt better than the other? 

Finally, did you experience that GPT-2 produces a new result everytime you run the chunk? Do you remember why that is?

<details>
<summary>ANSWER</summary>
Flan-T5 is seems better than GPT-2, but how you write the prompt also seems to affect performance.
<br><br>
As you might have noticed while playing with the demo, when we run GPT-2 with temperature sampling above 0, we introduce some stochasticity in the mix. That is, it will not always generate the most probable next token. Instead, it can produce a wide variety of outputs.
</details>
:::

#### My Solution

In [15]:
# two prompts
task_prefix = "summarize this"
prompt = f"{task_prefix}: {student_text}"

t5 = pipeline_t5(prompt)
gpt = pipeline_gpt(prompt)

print("PROMPT TASK PREFIX:", task_prefix)
print("Flan T5 Summary:", t5[0]['generated_text'])
print("\n")
print("GPT-2 Summary:", gpt[0]['generated_text'])
print("\n")

# make a summary of this:
task_prefix = "write a summary of this"
prompt = f"{task_prefix}: {student_text}"

t5 = pipeline_t5(prompt)
gpt = pipeline_gpt(prompt)

print("PROMPT TASK PREFIX:", task_prefix)
print("Flan T5 Summary:", t5[0]['generated_text'])
print("\n")
print("GPT-2 Summary:", gpt[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


PROMPT TASK PREFIX: summarize this
Flan T5 Summary: Online-teaching should be widely used among colleges and universities in terms of convenience and optimizing cost for educational institutions.


GPT-2 Summary: This article was originally published at Bancroft and is reproduced here with permission.
This article is part of a series on the growing use of distance-learning as an effective way of learning.




Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


PROMPT TASK PREFIX: write a summary of this
Flan T5 Summary: Online-teaching should be widely used among colleges and universities in terms of convenience and optimizing cost for educational institutions. However, opponents of online-teaching would argue that attending lectures in person provides students an opportunity to communicate with teachers and other classmates.


GPT-2 Summary: The issue of how to best promote online-degrees and how to best promote classroom communication is a subject which will likely be explored in the next issue of The Philosophy of Education.
This issue will be covered in a future issue of The Philosophy of Education.


## 1.5 Translation (and using few-shot learning!)
Let's try another task, *translation*... We could structure a prompt like this:

> *"English: I've believed as many as six impossible things before breakfast. Danish: "*

This is considered a *zero-shot* prompt, as we are not giving any examples. We could also add some successful examples of English to Danish before asking this task to do *few-shot prompting*: 
```{figure} ../figures/class6/claude-finetuning-in-context-learning.jpg
---
name: zero-shot-few-shot-fine-tuning overview
width: 100%
---
Overview of in-context learning (prompting) versus fine-tuning. Re-interpretation of a figure by Brown et al. ([2020](https://arxiv.org/abs/2005.14165)) (Original paper on GPT-3!!)
```

### Your Turn: Experimenting with Translation
:::{admonition} HANDS-ON
:class: red
Translate a chosen sentence from English to another language that you know. Try structuring different prompts:
- Is there are difference in how you phrase the instruction? Does the word "translate" help? 
- Try **zero-shot** versus **few-shot** prompting - do succesful translation examples help the models?

*As an extra bonus if you or your group knows many languages*, try to see if the models are better at translating to one language over another!
:::